# Mutual Information-based Registration

## Introduction
In this report, we will implement 3D rigid image registration using mutual information. We will use a T1-weighted and a T2-weighted MRI scan of the same patient, where the patient lies in slightly different positions between the two image acquisition protocols. Because the contrasts between the two MRI protocols are different, we will use mutual information, which measures the difference between the distributions of the intensities of the two scans.

### Input data and code hints
Import Python libraries:

In [1]:
%pip install numpy nibabel scipy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from numpy import linalg as la
import matplotlib.pyplot as plt
import scipy as mpl
from mpl_toolkits import mplot3d
#%matplotlib notebook
%matplotlib tk
plt.ion()
np.set_printoptions( suppress=True )
import nibabel as nib
import scipy
from scipy.ndimage import map_coordinates

Read the two 3D scans you'll be working with in this exercise:

In [3]:
T1_fileName = 'IXI014-HH-1236-T1.nii.gz'
T2_fileName = 'IXI014-HH-1236-T2_rotated.nii.gz'
T1 = nib.load( T1_fileName )
T2 = nib.load( T2_fileName )
T1_data = T1.get_fdata()
T2_data = T2.get_fdata()

Below is code to define a simple interactive viewer class that can be used to visualize 2D cross-sections of a 3D array along three orthogonal directions. It takes a 3D volume as input and shows the location a "linked cursor" in all three cross-sections.

The initial location of the cursor is in the middle of the volume in each case. It can be changed by clicking on one of the cross-sections. The viewer also displays the voxel index $\mathbf{v}$ of the cursor.

In [4]:
class Viewer:
    def __init__(self, data ):
        self.fig, self.ax = plt.subplots()
        self.data = data
        self.dims = self.data.shape
        self.position = np.round( np.array( self.dims ) / 2 ).astype( int )
        self.draw()
        self.fig.canvas.mpl_connect( 'button_press_event', self )
        self.fig.show()

    def __call__(self, event):
        print( 'button pressed' )
        if event.inaxes is None: return
      
        x, y = round( event.xdata ), round( event.ydata )

        #
        if ( x > (self.dims[0]-1) ) and ( y <= (self.dims[1]-1) ): return # lower-right quadrant
          
        #
        if x < self.dims[0]:
          self.position[ 0 ] = x
        else:
          self.position[ 1 ] = x - self.dims[0]
        
        if y < self.dims[1]:
          self.position[ 1 ] = y
        else:
          self.position[ 2 ] = y -self.dims[1]
        
        print( f"  voxel index: {self.position}" )
        print( f"  intensity: {self.data[ self.position[0], self.position[1], self.position[2] ]}" )

        self.draw()

    def draw( self ):
        #
        # Layout on screen is like this:
        #
        #     ^            ^
        #  Z  |         Z  |
        #     |            |
        #     ----->        ---->  
        #       X             Y
        #     ^
        #  Y  |
        #     |
        #     ----->  
        #       X
        #
        dims = self.dims
        position = self.position
        
        xySlice = self.data[ :, :, position[ 2 ] ]
        xzSlice = self.data[ :, position[ 1 ], : ]
        yzSlice = self.data[ position[ 0 ], :, : ]
        
        kwargs = dict( vmin=self.data.min(), vmax=self.data.max(), 
                       origin='lower', 
                       cmap='gray',
                       picker=True )

        self.ax.clear()

        self.ax.imshow( xySlice.T, 
                        extent=( 0, dims[0]-1, 
                                 0, dims[1]-1 ), 
                        **kwargs )
        self.ax.imshow( xzSlice.T, 
                        extent=( 0, dims[0]-1, 
                                 dims[1], dims[1]+dims[2]-1 ), 
                        **kwargs )
        self.ax.imshow( yzSlice.T, extent=( dims[0], dims[0]+dims[1]-1, 
                                            dims[1], dims[1]+dims[2]-1 ), 
                        **kwargs )

        color = 'g'
        self.ax.plot( (0, dims[0]-1), (position[1], position[1]), color )
        self.ax.plot( (0, dims[0]+dims[1]-1), (dims[1]+position[2], dims[1]+position[2]), color )
        self.ax.plot( (position[0], position[0]), (0, dims[1]+dims[2]-1), color )
        self.ax.plot( (dims[0]+position[1], dims[0]+position[1]), (dims[1]+1, dims[1]+dims[2]-1), color )

        self.ax.set( xlim=(1, dims[0]+dims[1]), ylim=(0, dims[1]+dims[2]) )

        self.ax.text( dims[0] + dims[1]/2, dims[1]/2, 
                      f"voxel index: {position}",  
                      horizontalalignment='center', verticalalignment='center' )
  
        self.ax.axis( False )

        self.fig.canvas.draw()

### Task 1: Resample the T2-weighted scan to the image grid of the T1-weighted scan

To find the voxel-to-voxel transformation between $\mathbf{v}_{T1}$ and $\mathbf{v}_{T2}$, we will use:

$$
\begin{pmatrix} \mathbf{ v_{T2}} \\ 1 \end{pmatrix} = \mathbf{M}_{T2}^{-1} \cdot \mathbf{M}_{T1} \cdot \begin{pmatrix} \mathbf{ v_{T1}} \\ 1 \end{pmatrix}
.
$$
At the location $\mathbf{v}_{T2}$, you should then use cubic B-spline interpolation to determine the intensity in the T2-weighted scan, and store it at index $\mathbf{v}_{T1}$ in the newly created image.

In [5]:
M_T1 = T1.affine
M_T2 = T2.affine

M_T1_to_T2 = la.inv( M_T2 ) @ M_T1
print( "Voxel-to-voxel transformation T1 -> T2:\n", M_T1_to_T2 )

V1, V2, V3 = np.meshgrid( np.arange(T1_data.shape[0]), np.arange(T1_data.shape[1]), np.arange(T1_data.shape[2]), indexing='ij')

V_T1 = np.vstack( ( V1.ravel(), V2.ravel(), V3.ravel(), np.ones( V1.size ) ) )

V_T2 = ( M_T1_to_T2 @ V_T1 )[ :3, : ]

T2_data_resampled = map_coordinates( T2_data, V_T2, order=3, mode='constant', cval=0.0 )
T2_data_resampled = T2_data_resampled.reshape( T1_data.shape )

print("T1 shape:", T1_data.shape )
print("Resampled T2 shape:", T2_data_resampled.shape )

Voxel-to-voxel transformation T1 -> T2:
 [[  0.77544818  -0.16582521  -0.86816572 102.46437994]
 [ -0.68434491   0.0206479   -1.00794843 290.45750967]
 [  0.02500879   0.18590633  -0.02157998 -13.60976798]
 [  0.           0.           0.           1.        ]]
T1 shape: (256, 256, 150)
Resampled T2 shape: (256, 256, 150)


We can then visualize the T1-weighted and resampled T2-weighted volumes as follows:

In [6]:
Viewer( T2_data_resampled / T2_data_resampled.max() + T1_data / T1_data.max() )

![image](./Figure_1.png)

### Discussion & rotation axis

The images suggest that the resampling was done successfully. This is because the anatomical parts shown in the MRI appear to be at a similar scale, and they are not stretched out relative to each other. On the other hand, we can clearly see, especially in the top left image, that the T1w and T2w MRIs are positioned at an angle relative to each other. This will later be fixed in the registration step.

In addition, the top left image shows that the rotation was done in the Z-axis of the plane (using the coordinate system convention from below).

![image](./coordinate_system.png)

### Task 2: Compute and visualize the joint histogram

In this task, we will compute the joint histogram 
$$ 
\mathbf{H} = 
\begin{pmatrix}
h_{1,1} & ... & h_{1,B} \\
\vdots & ... & \vdots \\
h_{B,1} & ... & h_{B,B} \\
\end{pmatrix} 
$$
of the T1-weighted scan and the resampled T2-weighted scan you just created, using $B=32$ bins. In order not to let the registration process be dominated by the (extremely numerous) background voxels, we ignore all voxels with an intensity lower than 10 in either image when computing the histogram. Visualize the joint histogram using a 3D bar plot, and label the axes.

In [7]:
# Flatten 3D arrays to 1D
t1_flat = T1_data.ravel()
t2_flat = T2_data_resampled.ravel()

# Exclude background voxels (intensity < 10)
mask = (t1_flat >= 10) & (t2_flat >= 10)
t1_valid = t1_flat[mask]
t2_valid = t2_flat[mask]

# Compute the joint histogram with B = 32 bins
B = 32
H, xedges, yedges = np.histogram2d(t1_valid, t2_valid, bins=B)

# Generate positions and dimensions for the 3D bars
xpos, ypos = np.meshgrid(np.arange(B), np.arange(B), indexing="ij")
xpos = xpos.ravel()
ypos = ypos.ravel()
zpos = np.zeros_like(xpos)

dx = dy = 0.75
dz = H.ravel()

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
ax.bar3d(xpos, ypos, zpos, dx, dy, dz, color="darkcyan", shade=True)

ax.set_xlabel("T1 Intensity Bin", fontsize=11, labelpad=10)
ax.set_ylabel("Resampled T2 Intensity Bin", fontsize=11, labelpad=10)
ax.set_zlabel("Voxel Frequency", fontsize=11, labelpad=10)
ax.set_title("Joint Histogram Before Registration (B = 32)", fontsize=13)

plt.tight_layout()
plt.show()

![image](./Figure_2.png)

### Task 3: Compute the mutual information between the two images
In this task, we will write a function that takes two image volumes defined on the same image grid as input, and returns the mutual information between the two images:

$$ 
MI = H_{F} + H_{M} - H_{F,M} 
$$
with
$$ 
H_{F,M} = - \sum_{f=1}^{B} \sum_{m=1}^{B} p_{f,m} \log(p_{f,m}),
$$
$$ 
H_{F} = - \sum_{f=1}^{B} p_{f} \log(p_{f}), 
$$
and
$$
H_{M} = - \sum_{m=1}^{B} p_{m} \log(p_{m})
.
$$

In [8]:

def data_to_histogram(data1, data2, nb_bins, level):

    data1_flat = data1.ravel()
    data2_flat = data2.ravel()

    mask = (data1_flat >= level) & (data2_flat >= level)
    data1_valid = data1_flat[mask]
    data2_valid = data2_flat[mask]

    H, _, _ = np.histogram2d(data1_valid, data2_valid, bins=nb_bins)

    return H



def mutual_information(data1, data2, nb_bins, level) -> int:

    H = data_to_histogram(data1, data2, nb_bins, level)

    total_voxels = np.sum(H)

    # Normalized joint probability distribution p_{f,m}
    p_f_m = H / total_voxels

    # p_f sums across columns (axis=1, fixed image bins)
    # p_m sums across rows (axis=0, moving image bins)
    p_f = np.sum(p_f_m, axis=1)
    p_m = np.sum(p_f_m, axis=0)

    eps = 1e-12
    H_FM = -np.sum(p_f_m * np.log(p_f_m + eps))
    H_F = -np.sum(p_f * np.log(p_f + eps))
    H_M = -np.sum(p_m * np.log(p_m + eps))

    # Mutual Information
    MI = H_F + H_M - H_FM

    return MI


print("Mutual information between T1-weighted image and the resampled T2-weighted image:", mutual_information(T1_data, T2_data_resampled, 32, 10))

Mutual information between T1-weighted image and the resampled T2-weighted image: 0.11674745729049985


### Task 4: Evaluate the mutual information across a range of rotation angles

Next, we will implement a grid search over a range of rotation angles, with the goal of identifying the angle with approximately the highest mutual information (i.e., where the registration is best). Make an educated guess of a suitable range by visually inspecting the images (or by simply trying a few rotation angles and inspecting the result), and then define a list of candidate angles at intervals of e.g., 5° apart.

We loop over all candidate rotation angles, each time (1) creating a corresponding rotation matrix $\mathbf{R}$; (2) resampling the T2-weighted volume accordingly; (3) calculating the mutual information with the function you created above; and (4) storing the mutual information value.

A 3D rotation matrix can be parameterized as follows:

$$
\mathbf{R} = \mathbf{R}_z \mathbf{R}_y \mathbf{R}_x,
$$

where

$$
\mathbf{R}_z =
\begin{pmatrix}
 \cos\theta_z & -\sin\theta_z & 0 \\
 \sin\theta_z & \cos\theta_z & 0 \\
 0 & 0 & 1
\end{pmatrix}
$$

implements a rotation around the z-axis in world coordinates;

$$
\mathbf{R}_y =
\begin{pmatrix}
 \cos\theta_y & 0 & \sin\theta_y \\
 0 & 1 & 0 \\
 -\sin\theta_y & 0 & \cos\theta_y
\end{pmatrix}
$$

is a rotation around the y-axis; and

$$
\mathbf{R}_x =
\begin{pmatrix}
 1 & 0 & 0 \\
 0 & \cos\theta_x & -\sin\theta_x \\
 0 & \sin\theta_x & \cos\theta_x
\end{pmatrix}
$$

rotates around the x-axis.


In [11]:
import numpy as np

def create_rotation_matrix_4x4(angle_deg, axis='z'):
    """
    Returns a 4x4 homogeneous rotation matrix for a given angle (in degrees)
    around the specified axis ('x', 'y', or 'z').
    """
    rad = np.radians(angle_deg)
    cos_a = np.cos(rad)
    sin_a = np.sin(rad)
    
    M_R = np.eye(4)
    
    if axis == 'z':
        M_R[:3, :3] = np.array([
            [ cos_a, -sin_a, 0.0],
            [ sin_a,  cos_a, 0.0],
            [   0.0,    0.0, 1.0]
        ])
    elif axis == 'y':
        M_R[:3, :3] = np.array([
            [ cos_a, 0.0, sin_a],
            [   0.0, 1.0,   0.0],
            [-sin_a, 0.0, cos_a]
        ])
    elif axis == 'x':
        M_R[:3, :3] = np.array([
            [1.0,   0.0,    0.0],
            [0.0, cos_a, -sin_a],
            [0.0, sin_a,  cos_a]
        ])
    else:
        raise ValueError(f"Invalid axis: {axis}. Choose 'x', 'y', or 'z'.")
        
    return M_R


# 1. Define candidate angles from -30 to +30 degrees in 5-degree steps
candidate_angles = np.arange(-180, 180, 5)

# 2. Build a list or dictionary of matrices for the grid search
rotation_matrices = {
    angle: create_rotation_matrix_4x4(angle, axis='z') 
    for angle in candidate_angles
}

In [12]:
# 1. Reuse or reconstruct the homogeneous coordinate grid of T1 (from Task 1)
V1, V2, V3 = np.meshgrid(
    np.arange(T1_data.shape[0]),
    np.arange(T1_data.shape[1]),
    np.arange(T1_data.shape[2]),
    indexing='ij'
)
coords_T1_hom = np.vstack((V1.ravel(), V2.ravel(), V3.ravel(), np.ones(V1.size)))

# Base matrices
inv_M_T2 = la.inv(T2.affine)
M_T1 = T1.affine

# 2. Grid search loop
mi_results = {}

for angle, M_R in rotation_matrices.items():
    # Voxel-to-voxel mapping under candidate rotation in world space:
    # v_T2 = M_T2^(-1) @ M_R @ M_T1 @ v_T1
    M_v2v_candidate = inv_M_T2 @ M_R @ M_T1

    # Map T1 voxel grid coordinates into T2 space
    coords_T2 = (M_v2v_candidate @ coords_T1_hom)[:3, :]

    # Resample T2 scan using cubic B-spline interpolation
    T2_resampled = map_coordinates(
        T2_data, coords_T2, order=3, mode='constant', cval=0.0
    ).reshape(T1_data.shape)

    # Compute Mutual Information using your Task 3 function
    mi = mutual_information(T1_data, T2_resampled, nb_bins=32, level=10)
    mi_results[angle] = mi

    print(f"Angle: {angle:3d}° -> MI: {mi:.5f}")

Angle: -180° -> MI: 0.09665
Angle: -175° -> MI: 0.10038
Angle: -170° -> MI: 0.10145
Angle: -165° -> MI: 0.10316
Angle: -160° -> MI: 0.10433
Angle: -155° -> MI: 0.10131
Angle: -150° -> MI: 0.10167
Angle: -145° -> MI: 0.10461
Angle: -140° -> MI: 0.10797
Angle: -135° -> MI: 0.10956
Angle: -130° -> MI: 0.10891
Angle: -125° -> MI: 0.10275
Angle: -120° -> MI: 0.09908
Angle: -115° -> MI: 0.09856
Angle: -110° -> MI: 0.10065
Angle: -105° -> MI: 0.10171
Angle: -100° -> MI: 0.10229
Angle: -95° -> MI: 0.10225
Angle: -90° -> MI: 0.10114
Angle: -85° -> MI: 0.09948
Angle: -80° -> MI: 0.09670
Angle: -75° -> MI: 0.09459
Angle: -70° -> MI: 0.09359
Angle: -65° -> MI: 0.09282
Angle: -60° -> MI: 0.09126
Angle: -55° -> MI: 0.08985
Angle: -50° -> MI: 0.09001
Angle: -45° -> MI: 0.08957
Angle: -40° -> MI: 0.09095
Angle: -35° -> MI: 0.09179
Angle: -30° -> MI: 0.09239
Angle: -25° -> MI: 0.09470
Angle: -20° -> MI: 0.09917
Angle: -15° -> MI: 0.10171
Angle: -10° -> MI: 0.10646
Angle:  -5° -> MI: 0.11177
Angle:   0°

Visual inspection of the overlaid volumes in Task 1 indicates that the simulated misalignment between the $T_1$ and $T_2$ scans is primarily an in-plane rotation within the axial slices ($xy$-plane). 
Per the assignment hint confirming the movement is restricted to a single axis, the transformation simplifies from a full 3D rotation space to a one-dimensional parameter search around the world $z$-axis:

Full 3D rotation: $\mathbf{R} = \mathbf{R}_z(\theta_z) \mathbf{R}_y(\theta_y) \mathbf{R}_x(\theta_x)$ with $\theta_x \approx 0$ and $\theta_y \approx 0$.   
Search space: $\mathbf{R}(\theta) = \mathbf{R}_z(\theta)$ over $\theta \in [-30^\circ, 30^\circ]$ sampled in discrete steps of $\Delta\theta = 5^\circ$.   

A coupled multi-axis grid search across $(\theta_x, \theta_y, \theta_z)$ would require $N^3$ evaluations, making cubic B-spline interpolation computationally prohibitive over the full $256 \times 256 \times 150$ grid without necessity. 

### Task 5: Perform automatic registration

Plot the negative mutual information, i.e. the energy function $E(\mathbf{w})= H_{F,M}-H_{F}-H_{M}$ for every angle in your grid search space. Select the one with the lowest energy (i.e., the best angle for registration) and transform the T2-weighted volume according to this angle.

Use the Viewer() function to visualize your registration result.

In [13]:
# Angles and MI values from the Task 4 dictionary, sorted by angle
angles = np.array(sorted(mi_results.keys()))
mi_vals = np.array([mi_results[a] for a in angles])

# Energy = negative mutual information
energy = -mi_vals
best_angle = angles[np.argmin(energy)]
print(f"Best angle: {best_angle}°  (E = {energy.min():.4f}, MI = {mi_vals.max():.4f})")

plt.figure(figsize=(8, 4))
plt.plot(angles, energy, 'o-', markersize=3)
plt.axvline(best_angle, color='r', linestyle='--', label=f'minimum at {best_angle}°')
plt.xlabel(r'rotation angle $\theta_z$ (degrees)')
plt.ylabel(r'$E(\theta_z) = -MI$')
plt.title('Energy function over the grid search')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('Figure_energy.png', dpi=150)
plt.show()

Best angle: 45°  (E = -0.3510, MI = 0.3510)


In [14]:
# Final resampling at the best angle, with cubic B-spline interpolation
M_R_best = create_rotation_matrix_4x4(best_angle, axis='z')
coords_T2_best = (inv_M_T2 @ M_R_best @ M_T1 @ coords_T1_hom)[:3, :]

T2_registered = map_coordinates(T2_data, coords_T2_best, order=3,
                                mode='constant', cval=0.0).reshape(T1_data.shape)

mi_before = mutual_information(T1_data, T2_data_resampled, nb_bins=32, level=10)
mi_after  = mutual_information(T1_data, T2_registered,     nb_bins=32, level=10)
print(f"MI before registration: {mi_before:.4f}")
print(f"MI after registration:  {mi_after:.4f}")

# Overlay, as in Task 1, and save it for the report
v = Viewer(T2_registered / T2_registered.max() + T1_data / T1_data.max())
v.fig.savefig('Figure_registered.png', dpi=150)

MI before registration: 0.1167
MI after registration:  0.3510


#### Task 5: Explanation

**Energy function.** Registration is formulated as the minimization of the energy

$$
E(\theta_z) = H_{F,M} - H_F - H_M = -MI(\theta_z),
$$

i.e. the negative mutual information between the T1-weighted image (fixed image $F$) and the T2-weighted image (moving image $M$) after rotating the latter by $\theta_z$ around the world $z$-axis. Minimizing $E$ is equivalent to maximizing the mutual information, i.e. finding the rotation for which the intensities of one image are most predictive of the intensities of the other. The joint entropy $H_{F,M}$ is low when the joint histogram is concentrated in a few bins, which happens when corresponding tissues overlap. The marginal entropies $H_F$ and $H_M$ prevent trivial solutions in which the images only overlap in regions without content.

The figure below shows $E(\theta_z)$ for all candidate angles of the grid search in Task 4.

![image](./Figure_energy.png)

**Results.** The energy has a single, sharp minimum at $\theta_z = [45]^\circ$ ($E = [-0.38]$), while for most other angles the energy lies between roughly $[-0.15]$ and $[-0.08]$. The minimum is narrow: already $5^\circ$ away from the optimum the energy is considerably higher. A small local minimum is visible at $\theta_z = [-135]^\circ$, exactly $180^\circ$ away from the optimum. At this angle the head is turned back-to-front, and because the outline of the head is roughly symmetric, the skull and brain boundaries partially overlap again. Since the grid search evaluates the full range of angles, it cannot get stuck in such a local minimum, which could happen with a local optimization method started close to it.

**Registration result.** The T2-weighted volume was resampled onto the T1 grid using the rotation $\mathbf{R}_z([45]^\circ)$ and the voxel-to-voxel mapping

$$
\begin{pmatrix} \mathbf{v}_{T2} \\ 1 \end{pmatrix}
= \mathbf{M}_{T2}^{-1} \cdot \mathbf{M}_{R} \cdot \mathbf{M}_{T1} \cdot
\begin{pmatrix} \mathbf{v}_{T1} \\ 1 \end{pmatrix},
$$

with cubic B-spline interpolation. The mutual information increased from $[0.117]$ before registration to $[0.35]$ after registration.

![image](./Figure_registered.png)

Compared with the overlay in Task 1, the two head outlines in the axial view now coincide, and the double contours of the skull and scalp have disappeared, so the images are well aligned. The remaining differences are not caused by misalignment: the straight edges in the overlay mark the limits of the T2 field of view (voxels outside it were assigned intensity 0), and the T2 structures look blurrier because the T2 scan was acquired with much thicker slices than the T1 scan ($28$ vs. $150$ slices).

**Limitations.** The optimal angle is only determined up to the step size of the grid ($5^\circ$). A finer search (e.g. $1^\circ$ steps) around the optimum could refine the estimate. Furthermore, the transformation model only allows a rotation around the world origin and around a single axis; any translation or rotation around another axis could not be corrected with this one-parameter search. In this case the visual result indicates that the rotation around the $z$-axis was sufficient.

### Task 6: Compare the joint histogram before and after registration

Visualize the joint histogram after registration. Compare the result to your initial joint histogram (i.e., before registration) and discuss your findings in your report.

### Additional task for the enthusiastic student: Compare the MI obtained with landmark-based and intensity-based registration

Perform landmark-based registration between the T1-weighted and T2-weighted volumes instead, just like you did in the last exercise. For the resulting registration, view your results with the Viewer() class; visualize the joint histogram, and calculate the mutual information. 

Compare the results with those obtained using mutual information, and discuss.

> ***Hint:***
> - You can reuse the landmarks that you collected in the landmark-based registration exercise, as these were defined in voxel space. 

## Conclusion